In [1]:
from IPython.display import HTML
with open("custom/style.css", "r") as f:
    custom_style = f.read()
hide_cell_style = "<style>.jp-Cell:has(.hide-me) { display: yes; }</style>"
HTML(f"<div class='hide-me'></div><style>{custom_style}</style>{hide_cell_style}")

<h1><center><font color='blue'>Exercise: Analyze the MPI ray tracer</font></center></h1>

## Description

Ray tracing is a flexible algorithm for computer-aided image generation that can be highly parallelized. A C version of the ray tracer program with MPI parallelization can be found in the folder `RayTracer`. The central function is `calc_tile()`, which computes one tile of the picture. The size of one tile and of the whole picture must be provided at run time as arguments of  the program. Note that the code assumes that the picture size is a multiple of the tile size. If not, then the program stops with an error message. The program is parallelized using the <font color='green'>master-worker</font> paradigm in which the master process distributes work to worker processes and collects their results. Worker processes do not communicate with each other.

<img src="figs/raytracer.png" alt="raytracer.png" style="float: right; margin-right: 50px;" width="60%">
<!--```{image} figs/raytracer.png
:width: 800px
```-->


This exercise has two parts. The first part concerns the efficiency of the parallelization with respect to the tile size for a given picture size. The second part is an illustration of tools for performance analysis of an MPI program. Unfortunately, the latter cannot be done by you during the course because one cannot run graphic user interface on compute nodes and inside the Jupyterhub you have access only to a compute node, not a frontend node. Therefore, the second part will be presented by us and you can try it later on another machine where you have access to the tool and you can execute it.
<div style="text-align: center; font-size: 32px; max-width: 2000px;">
    <span">
        Part I
    </span>
</div> <p></p>

Compile it by loading the intel and intelmpi modules:
```bash
module load intel intelmpi itac
```
and then simply typing `make` and run it like:
```bash
mpirun -n 72 ./ray_par size tilesize
```
For this part of exercise, we suggest you to use `size=24000` (namely an image of $24000\times 24000$). The program outputs its runtime and its performance in million pixels per second (MPixel/s). If `tilesize` is too small or too large, then its performance will be poor. There is an optimal `tilesize` for this image size and you can find an approximate value if you run the program for different values of `tilesize`. We propose the following values `10, 12, 16, 20, 40, 50, 60, 100, 200, 400, 500, 600, 800, 1000, 1200, 1500, 2000` for the `tilesize`.
- What is the optimal value of `tilesize`?
- Why is the program inefficient for too small tilesize?
- Why is the program inefficient for too large tilesize?

<div style="text-align: center; font-size: 32px; max-width: 2000px;">
    <span">
        Part II
    </span>
</div> <p></p>

To build an instrumented binary that is ready for running and analysis with <font color='green'>ITAC</font>, you should modify the `Makefile` and add `-trace` to the compiler option, both to `CFLAGS` and `LDFLAGS`. As mentioned above, this part of the exercise cannot be done on a compute node. Take an ITAC trace of the run as shown in the lecture (details see below) with `tilesize=10`, `tilesize=100`, and `tilesize=1000`. **We recommend** a smaller value of `size` than that of **Part I** to reduce the trace content, so `size=10000` and on **72** cores or fewer.
- Look at the traces: Can you find obvious problems to justify low and high performance for different values of `tilesize`?

<div style="text-align: center; font-size: 32px; max-width: 2000px;">
    <span">
        Guidelines for a simple run with ITAC
    </span>
</div> <p></p>

```
Profiling with Intel Trace Analyzer and Collector (ITAC)
https://moodle.nhr.fau.de/mod/resource/view.php?id=1937
```

1. Load the ITAC module and export environment variables
```bash
module load intel intelmpi itac
export VT_FLUSH_PREFIX=/tmp
export VT_LOGFILE_FORMAT=SINGLESTF
```
2. Build the code (C version only) on the frontend (this does not work on a compute node):
```bash
make
```
3. After running the program, it creates a trace file, named as `ray_par.single.stf`
4. View the resulting trace file with ITAC:
```bash
traceanalyzer ray_par.single.stf
```
Remember that you will have to connect to the cluster with X forwarding enabled (`-X` or `-Y` on the `ssh` command line) so that the GUI can be displayed on your end.